In [1]:
import rasterio
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
import joblib

print("جميع المكتبات تعمل بنجاح ✅")

جميع المكتبات تعمل بنجاح ✅


In [2]:

# محاولة فتح الصورة
img_path = "..\shp\main_raster.tif"
with rasterio.open(img_path) as src:
    print("الصورة مفتوحة بنجاح ✅")
    print(f"عدد الباندات: {src.count}")
    print(f"أبعاد الصورة: {src.width} x {src.height}")

الصورة مفتوحة بنجاح ✅
عدد الباندات: 10
أبعاد الصورة: 309 x 161


In [3]:
# اختيار الباندات الثلاثة
# عادة Sentinel-2:
# B4 = Red, B3 = Green, B2 = Blue
bands = [4, 3, 2]  # [Red, Green, Blue]

with rasterio.open(img_path) as src:
    img_data = src.read(bands)  # سيصبح شكلها (3, height, width)

print(f"تم اختيار الباندات: {bands}")
print(f"شكل المصفوفة: {img_data.shape} (Bands, Height, Width)")

تم اختيار الباندات: [4, 3, 2]
شكل المصفوفة: (3, 161, 309) (Bands, Height, Width)


In [4]:
# إعادة تشكيل الصورة
height, width = img_data.shape[1], img_data.shape[2]
img_reshaped = np.moveaxis(img_data, 0, -1).reshape(-1, 3)  # الشكل الآن (num_pixels, 3)

print(f"تم إعادة تشكيل البيانات إلى جدول: {img_reshaped.shape} (عدد البكسلات, عدد الباندات)")

تم إعادة تشكيل البيانات إلى جدول: (49749, 3) (عدد البكسلات, عدد الباندات)


In [5]:
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [6]:
# مسار الصورة
raster_path = r"..\shp\main_raster.tif"

# مسار shapefile
shapefile_path = r"..\shp\training_point.shp"

# قراءة الصورة
src = rasterio.open(raster_path)

# قراءة shapefile
gdf = gpd.read_file(shapefile_path)
print(gdf.head())

   id                        geometry
0   3    POINT (459681.81 3319760.62)
1   3  POINT (459720.979 3319212.254)
2   3  POINT (459923.352 3318043.711)
3   3   POINT (461679.43 3320041.331)
4   3   POINT (459910.296 3317514.93)


In [7]:
# دالة لاستخراج قيم البكسل لكل نقطة
def extract_pixel_values(geom, src, bands=[4,3,2]):
    row, col = src.index(geom.x, geom.y)  # إحداثيات البكسل
    return [src.read(b)[row, col] for b in bands]

In [8]:
print("CRS before:", gdf.crs)
gdf = gdf.to_crs(src.crs)
print("CRS after:", gdf.crs)

CRS before: EPSG:32636
CRS after: PROJCS["WGS 84 / UTM zone 36N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",33],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32636"]]


In [10]:
with rasterio.open(raster_path) as src:
    gdf = gdf.to_crs(src.crs)
    height = src.height
    width = src.width
    bounds = src.bounds
    print("Bounds:", bounds, "Height:", height, "Width:", width)

    X, y = [], []

    for _, row in gdf.iterrows():
        if row.geometry.geom_type == 'Point':
            x, y_pt = row.geometry.x, row.geometry.y

            # تحقق أولاً من حدود raster
            if (bounds.left <= x <= bounds.right) and (bounds.bottom <= y_pt <= bounds.top):
                # تحويل الإحداثيات إلى بكسل
                col, row_pix = src.index(x, y_pt)

                # 🟡 طباعة معلومات التشخيص
                print(f"x={x:.2f}, y={y_pt:.2f} -> col={col}, row_pix={row_pix}, width={width}, height={height}")

                # تحقق من أن الإحداثيات داخل الصورة فعلياً
                if 0 <= row_pix < height and 0 <= col < width:
                    X.append(extract_pixel_values(row.geometry, src))
                    y.append(row['id'])
                else:
                    print(f"❌ النقطة بعد تحويلها لبكسل خارج الصورة: ({x:.2f}, {y_pt:.2f}) -> بكسل ({row_pix}, {col}) خارج النطاق")
            else:
                print(f"النقطة خارج bounds الصورة: ({x:.2f}, {y_pt:.2f})")


print(f"\nعدد العينات الصالحة: {len(X)}")
print("توزيع الفئات:")
print(pd.Series(y).value_counts())

Bounds: BoundingBox(left=456520.0, bottom=3317080.0, right=462700.0, top=3320300.0) Height: 161 Width: 309
x=459681.81, y=3319760.62 -> col=26, row_pix=158, width=309, height=161
x=459720.98, y=3319212.25 -> col=54, row_pix=160, width=309, height=161
x=459923.35, y=3318043.71 -> col=112, row_pix=170, width=309, height=161
❌ النقطة بعد تحويلها لبكسل خارج الصورة: (459923.35, 3318043.71) -> بكسل (170, 112) خارج النطاق
x=461679.43, y=3320041.33 -> col=12, row_pix=257, width=309, height=161
❌ النقطة بعد تحويلها لبكسل خارج الصورة: (461679.43, 3320041.33) -> بكسل (257, 12) خارج النطاق
x=459910.30, y=3317514.93 -> col=139, row_pix=169, width=309, height=161
❌ النقطة بعد تحويلها لبكسل خارج الصورة: (459910.30, 3317514.93) -> بكسل (169, 139) خارج النطاق
x=461960.14, y=3320113.14 -> col=9, row_pix=272, width=309, height=161
❌ النقطة بعد تحويلها لبكسل خارج الصورة: (461960.14, 3320113.14) -> بكسل (272, 9) خارج النطاق
x=462169.04, y=3319936.88 -> col=18, row_pix=282, width=309, height=161
❌ النقطة بع

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

# تقسيم البيانات
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# تحويل y_train و y_val إلى pandas Series
y_train = pd.Series(y_train)
y_val = pd.Series(y_val)

# التحقق من حجم كل مجموعة
print("عدد عينات التدريب:", len(X_train))
print("عدد عينات التحقق:", len(X_val))

# توزيع الفئات
print("\nتوزيع الفئات في التدريب:")
print(y_train.value_counts())
print("\nتوزيع الفئات في التحقق:")
print(y_val.value_counts())

عدد عينات التدريب: 58
عدد عينات التحقق: 15

توزيع الفئات في التدريب:
1    29
2    18
3    11
Name: count, dtype: int64

توزيع الفئات في التحقق:
1    7
2    5
3    3
Name: count, dtype: int64


In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

# 1️⃣ إنشاء نموذج Decision Tree
# max_depth=5 لتجنب overfitting، random_state=42 لضمان التكرار
model = DecisionTreeClassifier(max_depth=5, random_state=42)

# 2️⃣ تدريب النموذج على بيانات التدريب
model.fit(X_train, y_train)

# 3️⃣ توقع النتائج على بيانات التدريب
y_train_pred = model.predict(X_train)

# 4️⃣ توقع النتائج على بيانات التحقق
y_val_pred = model.predict(X_val)

# 5️⃣ حساب مقاييس الأداء للتدريب
print("=== أداء مجموعة التدريب ===")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("Precision:", precision_score(y_train, y_train_pred, average='weighted'))
print("Recall:", recall_score(y_train, y_train_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred))
print("\nClassification Report:\n", classification_report(y_train, y_train_pred))

# 6️⃣ حساب مقاييس الأداء للتحقق
print("=== أداء مجموعة التحقق ===")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, average='weighted'))
print("Recall:", recall_score(y_val, y_val_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))

=== أداء مجموعة التدريب ===
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
Confusion Matrix:
 [[29  0  0]
 [ 0 18  0]
 [ 0  0 11]]

Classification Report:
               precision    recall  f1-score   support

           1       1.00      1.00      1.00        29
           2       1.00      1.00      1.00        18
           3       1.00      1.00      1.00        11

    accuracy                           1.00        58
   macro avg       1.00      1.00      1.00        58
weighted avg       1.00      1.00      1.00        58

=== أداء مجموعة التحقق ===
Accuracy: 0.9333333333333333
Precision: 0.9416666666666667
Recall: 0.9333333333333333
Confusion Matrix:
 [[7 0 0]
 [1 4 0]
 [0 0 3]]

Classification Report:
               precision    recall  f1-score   support

           1       0.88      1.00      0.93         7
           2       1.00      0.80      0.89         5
           3       1.00      1.00      1.00         3

    accuracy                           0.93        15
   macro avg

In [18]:
import joblib
joblib.dump(model, "model.pkl")
print("تم حفظ النموذج بنجاح في model.pkl")

تم حفظ النموذج بنجاح في model.pkl
